# Phase 11 — Production Hardening, Security, Deployment Architecture and Observability

## 1. Phase Overview

Phase 11 focused on preparing VIGILOX for production-oriented operation.

By the end of Phase 10, the system already had a strong document-intelligence pipeline:

```text
Upload
  ↓
Duplicate Detection
  ↓
Durable Job
  ↓
Worker
  ↓
PaddleOCR
  ↓
Groq Structured Extraction
  ↓
Evidence Validation
  ↓
Quality Assessment
  ↓
Machine Decision
  ↓
Human Review
  ↓
Final Record
```

The next question was no longer only:

> Does the document intelligence logic work?

The new question became:

> Can the service run safely, predictably, and observably in a real deployment environment?

Phase 11 therefore concentrated on:

- reviewer identity security
- trusted proxy boundaries
- production startup validation
- request security
- security headers
- CORS policy
- rate limiting
- database connection management
- API concurrency
- containerization
- Nginx reverse proxy configuration
- structured logging
- operational metrics
- worker health
- database migrations
- backup and restore
- graceful shutdown
- deployment documentation
- release-readiness validation

The core principle of this phase was:

> Production readiness is not one feature. It is the combination of secure boundaries, predictable configuration, operational visibility, failure handling, and reproducible deployment behavior.

---

# 2. Objective

The primary objective of Phase 11 was:

> Harden VIGILOX so that the application could move from a development-oriented environment toward a controlled production deployment model.

The phase addressed the layers surrounding the document-processing pipeline:

```text
Users
  ↓
Reverse Proxy
  ↓
Security Boundary
  ↓
FastAPI
  ↓
PostgreSQL
  ↓
Worker
  ↓
OCR + LLM
```

This required the project to define not only application logic, but also:

- who can review documents
- which network sources can supply identity
- how requests are identified
- how load is bounded
- how services expose health
- how logs are structured
- how workers report liveness
- how schema changes are applied
- how data is backed up
- how services shut down safely
- how production deployment is documented

---

# 3. Problems Before Phase 11

Several important production concerns remained after Phase 10.

## 3.1 Development Reviewer Identity Was Not Production Authentication

During local development, a reviewer identity such as:

```text
local-reviewer
REVIEWER
```

was sufficient.

However, using the same identity for every production reviewer would destroy audit accountability.

The system needed a production identity boundary.

---

## 3.2 HTTP Headers Could Be Spoofed

A deployment using reviewer identity headers such as:

```text
X-VIGILOX-REVIEWER-ID
X-VIGILOX-REVIEWER-ROLE
```

cannot simply trust every request that contains them.

Any client that can reach the API port can send arbitrary HTTP headers.

Therefore trusted-header identity required a network trust boundary.

---

## 3.3 Production Misconfiguration Needed to Fail Closed

Some configuration mistakes are too dangerous to allow silently.

Examples:

```text
Production + local reviewer identity
```

or:

```text
trusted_headers mode
+
no trusted proxy configuration
```

A service that starts successfully with unsafe identity configuration may appear healthy while recording incorrect audit information.

The safer behavior is:

```text
Unsafe Production Configuration
        ↓
Startup Failure
```

---

## 3.4 API Concurrency and Database Pool Capacity Could Diverge

FastAPI synchronous routes run in worker threads.

If the application allowed more concurrent request threads than available database connections, requests could block waiting for connections.

This could create failures even when PostgreSQL itself was healthy.

---

## 3.5 Security Headers Needed Centralized Enforcement

The application needed consistent browser protections across all routes.

Without a shared security middleware layer, individual pages could accidentally receive different policies.

---

## 3.6 Operational Health Needed More Than `/health`

A running API process does not prove:

```text
Database reachable
Storage available
Worker alive
Queue processing
```

Health needed to be separated into meaningful signals.

---

## 3.7 Logs Needed Structure

Plain text debugging logs are useful during development.

Production operations require consistent fields that can be searched and correlated across requests and jobs.

---

## 3.8 Deployment Needed Reproducibility

A production-oriented system should not depend on manually remembering:

```text
which command starts the API
which command starts the worker
when migrations run
which port is public
which volumes contain documents
```

The deployment model needed to be explicitly represented in code and documentation.

---

# 4. Phase 11 Design Principles

## 4.1 Fail Closed

When security-sensitive configuration is incomplete, the application should reject the configuration rather than silently weaken controls.

```text
Unknown / Unsafe
      ↓
Reject
```

rather than:

```text
Unknown / Unsafe
      ↓
Allow
```

---

## 4.2 Server-Authoritative Identity

The browser is never authoritative for reviewer identity.

The frontend can display:

```text
Reviewer: Alice
Role: REVIEWER
```

but the server decides whether that identity is valid.

---

## 4.3 Trust Must Have a Boundary

If identity is supplied through headers, the system must know:

```text
Who is allowed to provide these headers?
```

Trusting the header value without trusting the sender is incomplete security.

---

## 4.4 Least Public Exposure

Only the component that must be public should be directly exposed.

The intended deployment model became:

```text
Internet
   ↓
Nginx
   ↓
FastAPI
```

rather than exposing the application server directly.

---

## 4.5 Health Signals Should Mean One Thing

Different operational questions should use different checks.

For example:

```text
Is the API process alive?
→ /health

Can the API serve normal traffic?
→ /health/ready

Is a worker available?
→ worker health
```

---

## 4.6 Configuration Should Be Explicit

Production behavior should be controlled through documented environment variables rather than hidden assumptions.

---

# 5. Reviewer Identity Architecture

The reviewer identity system supports two major modes:

```text
local_env
trusted_headers
```

---

## 5.1 Local Environment Identity

Local development can use:

```env
VIGILOX_REVIEW_IDENTITY_MODE=local_env
VIGILOX_LOCAL_REVIEWER_ID=local-reviewer
VIGILOX_LOCAL_REVIEWER_ROLE=REVIEWER
```

This is convenient for:

- development
- testing
- controlled local demonstrations

The reviewer identity is supplied by server configuration rather than the browser.

---

## 5.2 Trusted Header Identity

Production-oriented deployments can use:

```env
VIGILOX_REVIEW_IDENTITY_MODE=trusted_headers
```

In this mode, reviewer identity comes from headers:

```text
X-VIGILOX-REVIEWER-ID
X-VIGILOX-REVIEWER-ROLE
```

These headers are expected to be inserted by a trusted reverse proxy or identity-aware gateway.

---

# 6. Why Header Identity Alone Is Unsafe

Without sender validation:

```text
Client
  ↓
X-VIGILOX-REVIEWER-ID: administrator
X-VIGILOX-REVIEWER-ROLE: ADMIN
  ↓
API
```

would be indistinguishable from a trusted proxy request.

Therefore identity headers are only meaningful when the source of the request is also trusted.

---

# 7. Trusted Proxy Configuration

Trusted reverse proxy sources are configured with:

```env
VIGILOX_TRUSTED_PROXIES=10.0.0.0/8,172.16.0.0/12
```

The application accepts trusted identity behavior only from explicitly configured:

- IP addresses
- CIDR ranges

Conceptually:

```text
Request
   ↓
Source Address
   ↓
Trusted Proxy?
   ├── Yes → trusted identity headers may be used
   └── No  → headers are not authoritative
```

---

# 8. Reverse Proxy Header Stripping

The reverse proxy must remove client-supplied copies of identity headers before injecting trusted values.

Conceptually:

```text
Internet Client
     ↓
Attempts:
X-VIGILOX-REVIEWER-ROLE: ADMIN
     ↓
Nginx
     ↓
Remove Client Header
     ↓
Authenticate / Establish Identity
     ↓
Inject Trusted Header
     ↓
FastAPI
```

This prevents a browser from self-assigning reviewer privileges.

---

# 9. Defense in Depth

Header stripping alone is not sufficient.

Imagine:

```text
Nginx strips spoofed identity headers
```

but:

```text
FastAPI port is also publicly reachable
```

A client could bypass Nginx entirely.

Therefore Phase 11 combined:

```text
Proxy Header Stripping
        +
Application Trusted-Source Validation
```

Both controls are needed.

---

# 10. Production Startup Validation

A deployment environment variable was introduced:

```env
VIGILOX_ENVIRONMENT=production
```

Production mode enables additional startup validation.

---

## 10.1 Reject Local Reviewer Identity in Production

This configuration:

```text
VIGILOX_ENVIRONMENT=production
VIGILOX_REVIEW_IDENTITY_MODE=local_env
```

is rejected.

Reason:

```text
Every reviewer
     ↓
same configured identity
     ↓
audit history loses real attribution
```

---

## 10.2 Reject Untrusted Header Configuration

This configuration is also rejected:

```text
production
+
trusted_headers
+
no VIGILOX_TRUSTED_PROXIES
```

because the application would have no trustworthy way to distinguish proxy-supplied headers from arbitrary clients.

---

# 11. Reviewer Authorization

Reviewer roles include:

```text
VIEWER
REVIEWER
ADMIN
```

The backend enforces authorization for review operations.

The frontend may hide unavailable actions for usability, but that is not the security boundary.

Correct architecture:

```text
Browser
   ↓
Review Request
   ↓
Backend Identity
   ↓
Backend Authorization
   ↓
Action Allowed / Rejected
```

---

# 12. Server-Authoritative Request IDs

Phase 11 strengthened request correlation.

Each request receives a server-controlled request ID.

The response includes:

```text
X-Request-ID
```

The application's structured error envelope includes the same identifier.

Example:

```json
{
  "status": "error",
  "detail": "Safe description",
  "error": {
    "code": "ERROR_CODE",
    "message": "Safe message",
    "request_id": "..."
  }
}
```

---

# 13. Why Request IDs Are Server Controlled

A client-supplied identifier should not become the only trusted operational identifier.

The server must be able to reliably correlate:

```text
HTTP request
log events
error response
operational investigation
```

without trusting arbitrary user-controlled identifiers.

---

# 14. Security Headers

Phase 11 introduced centralized browser security headers.

Representative controls include:

```text
Content-Security-Policy
X-Content-Type-Options
X-Frame-Options
Referrer-Policy
Permissions-Policy
Cross-Origin-Opener-Policy
```

These headers are applied consistently rather than configured independently on each page.

---

# 15. Content Security Policy

Content Security Policy restricts the browser resources the application is allowed to load.

The application primarily uses same-origin:

```text
HTML
CSS
JavaScript
API requests
Document images
```

This enabled a restrictive security posture compared with applications that rely heavily on external scripts.

---

# 16. No Unsafe Inline Event Handlers

The frontend architecture avoids patterns such as:

```html
<button onclick="approveDocument()">Approve</button>
```

and prefers JavaScript event listeners.

This improves compatibility with restrictive CSP rules.

---

# 17. HSTS

HTTP Strict Transport Security support was added as a configurable production option.

Example:

```env
VIGILOX_HSTS_ENABLED=true
```

It is intentionally not enabled blindly.

HSTS should only be enabled when HTTPS is genuinely and consistently terminated in front of the service.

The reason is that browsers remember HSTS policies.

Incorrect HSTS configuration can make a host inaccessible over plain HTTP long after the original response was sent.

---

# 18. CORS Policy

The VIGILOX frontend and API normally share one origin.

Therefore:

```text
Browser Page Origin
      =
API Origin
```

and CORS is generally unnecessary.

The safest default is no cross-origin allowance.

If a deployment genuinely serves the frontend from another origin, exact origins may be configured.

Example:

```env
VIGILOX_CORS_ORIGINS=https://reviews.example.com
```

Wildcard production access is rejected.

---

# 19. Upload Rate Limiting

Document upload is expensive because it creates durable work.

A loop that continuously uploads files can create a backlog that survives the client connection.

Phase 11 therefore introduced application-level upload rate limiting.

Representative configuration:

```env
VIGILOX_RATE_LIMIT_ENABLED=true
VIGILOX_UPLOAD_RATE_LIMIT=30
VIGILOX_UPLOAD_RATE_WINDOW_SECONDS=60
```

---

# 20. Process-Local Rate Limit Semantics

The application-level limiter is intentionally described accurately.

It is:

```text
per application process
```

not:

```text
global distributed rate limit
```

If there are:

```text
3 API replicas
```

each process has its own in-memory budget.

Therefore deployment-level Nginx limiting provides an additional ingress control.

---

# 21. Request Concurrency

The application has synchronous route handlers.

FastAPI executes these in a worker thread pool.

Phase 11 explicitly controlled request concurrency with:

```env
VIGILOX_REQUEST_CONCURRENCY=20
```

The objective was to align the application's concurrent request capacity with database resources.

---

# 22. Database Connection Pool

Representative database configuration includes:

```env
VIGILOX_DB_POOL_SIZE=10
VIGILOX_DB_POOL_TIMEOUT_SECONDS=10
VIGILOX_DB_POOL_RECYCLE_SECONDS=1800
VIGILOX_DB_CONNECT_TIMEOUT_SECONDS=10
```

These settings prevent connection handling from relying entirely on library defaults.

---

# 23. Concurrency and Database Capacity Relationship

The important relationship is:

```text
API request concurrency
        ↓
Potential database sessions
```

For multiple API processes:

```text
Total Potential Connections
≈
API Processes × Request Capacity
+
Worker Processes
+
Operational Connections
```

Database capacity must be considered before adding more application replicas.

---

# 24. Short Database Pool Timeout

If application concurrency is properly bounded, waiting for a database connection should be unusual.

A relatively short pool timeout makes resource exhaustion visible quickly instead of allowing requests to hang for long periods.

---

# 25. Database Connection Recycling

Connection recycling helps handle:

- long-lived database connections
- load balancers
- database proxies
- deployments
- infrastructure that closes idle connections

Combined with connection pre-ping, this improves resilience to stale connections.

---

# 26. Database Connect Timeout

Without an explicit connection timeout:

```text
Database network unreachable
      ↓
OS-level TCP timeout
      ↓
request thread may block for a long time
```

An explicit database connect timeout makes failures bounded and allows readiness checks to respond meaningfully.

---

# 27. API Pipeline Initialization

Phase 11 formalized the different initialization needs of API and worker processes.

For the API:

```env
VIGILOX_API_EAGER_PIPELINE=false
```

is suitable for deployments using the async job API.

The API does not need to load PaddleOCR simply to:

- serve the dashboard
- accept job creation
- provide document lists
- handle review actions

---

# 28. Worker Pipeline Initialization

Workers normally use eager OCR initialization:

```text
Worker Starts
    ↓
PaddleOCR Loads
    ↓
Worker Ready
    ↓
Job Claimed
```

This provides two advantages:

1. model initialization occurs outside an active job lease
2. a broken OCR installation prevents the worker from claiming jobs

---

# 29. Database Migrations

Phase 11 formalized schema management with Alembic.

Typical commands:

```text
alembic current
alembic history
alembic upgrade head
alembic downgrade -1
```

Production schema changes are no longer expected to depend on manual table creation or ad-hoc ALTER statements.

---

# 30. Environment-Driven Alembic Configuration

Database credentials are not stored directly in:

```text
alembic.ini
```

Instead:

```text
migrations/env.py
```

reads:

```env
DATABASE_URL
```

from the environment.

This keeps:

```text
Application Database
        =
Migration Database
```

under one configuration source.

---

# 31. Why No Database Password in `alembic.ini`

`alembic.ini` is committed to source control.

A real database URL there would expose credentials.

A placeholder URL would create two separate configuration locations.

Using `DATABASE_URL` avoids both problems.

---

# 32. Docker Image

Phase 11 introduced a production-oriented Docker image.

The image was designed to support multiple operational roles from the same codebase.

Representative roles:

```text
api
worker
migrate
```

Conceptually:

```text
One Application Image
        ↓
Role Selected at Startup
        ├── API
        ├── Worker
        └── Migrations
```

This reduces version drift between application components.

---

# 33. Non-Root Container Execution

The application image uses a non-root runtime user.

Representative UID:

```text
10001
```

Running application services as root inside containers is unnecessary for normal VIGILOX operation.

---

# 34. OCR Model Preparation

The container build prepares the OCR runtime so production workers do not depend on unexpected model downloads after deployment.

This reduces runtime uncertainty.

The desired deployment behavior is:

```text
Build
   ↓
Dependencies / Models Ready
   ↓
Deploy
   ↓
Worker Starts
```

rather than downloading major runtime assets only when the first document arrives.

---

# 35. Docker Compose Architecture

The production-oriented Compose configuration models multiple services.

Conceptually:

```text
Nginx
  ↓
API
  ↓
PostgreSQL

Worker
  ↓
PostgreSQL

Migrate
  ↓
PostgreSQL
```

with separate persisted storage for:

```text
Managed Documents
Pending Uploads
Database Data
```

---

# 36. Reverse Proxy

Nginx acts as the public ingress boundary.

Its responsibilities include:

- reverse proxying
- request controls
- reviewer-header sanitization
- rate limiting
- security boundary enforcement
- routing

The API is not intended to be directly public in this architecture.

---

# 37. Separate Storage Volumes

Pending uploads and managed documents remain separated at deployment level.

Conceptually:

```text
Managed Volume
      ≠
Pending Volume
```

This preserves the storage invariants established during the async processing phase.

---

# 38. TLS Material

TLS certificate and key material is deliberately excluded from source control.

A private key committed to Git is no longer private.

The repository may contain documentation describing where runtime TLS material belongs, but not the actual secret material.

---

# 39. Structured Logging

Phase 11 introduced structured operational logging.

Instead of relying only on free-form text:

```text
Job completed successfully
```

the system can emit structured events such as:

```text
event=job.completed
job_state=COMPLETED
duration_ms=...
request_id=...
```

This makes logs easier to:

- search
- aggregate
- filter
- correlate

---

# 40. Logging Privacy

The logging design deliberately avoids recording full sensitive document contents.

Logs should not contain:

- OCR text
- extracted personal data
- API keys
- database passwords
- reviewer correction values
- raw managed filesystem paths

Operational logs should describe events without becoming a second copy of sensitive business data.

---

# 41. Representative Log Events

Examples include:

```text
worker.starting
worker.warmup_complete
job.retry_scheduled
job.completed
```

Stable event names make operational queries easier than relying on natural-language message parsing.

---

# 42. Operational Metrics

Phase 11 introduced metrics designed for operational monitoring.

Useful metric categories include:

- HTTP request counts
- HTTP request duration
- job queue depth
- job state counts
- completed jobs
- failed jobs
- worker processing duration
- OCR duration
- LLM duration
- provider rate-limit events
- batch results
- worker heartbeat state

---

# 43. Metrics Cardinality

Metrics must avoid uncontrolled labels such as:

```text
document_id
filename
reviewer_id
job_id
```

If every request produces a unique metric label, the monitoring system can become overwhelmed by cardinality.

Labels are therefore kept bounded and operationally meaningful.

---

# 44. Metrics Endpoint

The application supports a Prometheus-compatible endpoint:

```http
GET /metrics
```

Metrics exposure is configurable.

Representative setting:

```env
VIGILOX_METRICS_ENABLED=true
```

Production exposure should also be protected at the network/proxy layer.

---

# 45. API Liveness

The API liveness endpoint is:

```http
GET /health
```

Its responsibility is narrow:

> Is the API process alive and able to answer?

Example:

```json
{
  "status": "ok",
  "service": "vigilox-document-intelligence",
  "version": "0.1.0"
}
```

---

# 46. API Readiness

Readiness is exposed separately:

```http
GET /health/ready
```

Readiness checks operational dependencies such as:

- PostgreSQL connectivity
- storage availability
- critical service configuration
- database connection capacity

It does not perform expensive OCR or external LLM processing.

---

# 47. Why Readiness Does Not Call Groq

An external provider may temporarily rate-limit or fail while the application itself remains healthy.

If readiness depended on a live LLM request:

```text
Groq temporary outage
       ↓
API marked unhealthy
       ↓
deployment system may restart healthy application
```

This would confuse application health with provider availability.

---

# 48. Worker Health

Worker availability is monitored separately from API readiness.

Possible worker states include:

```text
HEALTHY
STALE
NO_WORKER
```

This distinction matters because:

```text
API Online
```

does not mean:

```text
Documents are being processed
```

---

# 49. Worker Heartbeats

Workers periodically write heartbeat information to PostgreSQL.

The system can then ask:

```text
When was the worker last seen?
```

A representative stale threshold is:

```env
VIGILOX_WORKER_STALE_AFTER_SECONDS=420
```

---

# 50. Worker Staleness and Job Lease

The worker stale threshold is intentionally greater than the job lease.

Representative relationship:

```text
Job Lease
360 seconds

Worker Stale Threshold
420 seconds
```

A worker legitimately processing a slow document should not automatically be classified as dead before the allowed processing window ends.

---

# 51. Queue Without Worker Signal

Another useful operational condition is:

```text
Jobs are waiting
+
No healthy worker
```

This condition can be surfaced immediately rather than waiting only for a stale-heartbeat timeout.

It directly answers:

> Is work waiting with nobody available to process it?

---

# 52. Backup Strategy

Phase 11 introduced backup and restore procedures for the system's related state.

Important data includes:

```text
PostgreSQL
Managed Document Storage
Pending Retryable Sources
```

A database backup alone is not the complete VIGILOX state.

---

# 53. Database and Filesystem Relationship

A document record and its source file are related.

Therefore:

```text
Database Dump at Time A
+
Unrelated Filesystem Copy at Time B
```

should not automatically be assumed to represent one perfectly consistent snapshot.

Backup procedures need to consider this relationship.

---

# 54. PostgreSQL Backup Tools

Operational scripts use standard PostgreSQL tools such as:

```text
pg_dump
pg_restore
```

An optional configuration can identify their installation directory:

```env
VIGILOX_PG_BIN=...
```

This is useful on systems where PostgreSQL client tools are not available on PATH.

---

# 55. PostgreSQL Client Version Awareness

Backup tooling needs compatible PostgreSQL client versions.

The backup process therefore documents client/server compatibility rather than assuming any installed `pg_dump` is suitable.

---

# 56. Graceful Shutdown

The API and worker need controlled shutdown behavior.

For the worker, shutdown should:

```text
Receive Stop Signal
      ↓
Stop Claiming New Jobs
      ↓
Preserve Durable State
      ↓
Release Resources
      ↓
Exit
```

The worker should not falsely mark an interrupted job as successfully completed.

---

# 57. Lease-Based Recovery After Shutdown

If a worker is interrupted during processing:

```text
Worker Stops
    ↓
No false completion
    ↓
Lease eventually expires
    ↓
Job becomes recoverable
```

This keeps recovery aligned with the durable job architecture.

---

# 58. Storage Safety Hardening

The storage layer retained and strengthened protections such as:

- canonical storage roots
- safe document identifiers
- path traversal rejection
- symlink rejection
- atomic saves
- database-first deletion
- missing-source detection
- orphan detection

These controls reduce the chance that malformed paths or filesystem state can corrupt managed document storage.

---

# 59. Database-First Deletion

When deleting a managed record, database state and filesystem state must be coordinated.

The storage model avoids blindly deleting files first and then attempting database changes afterward.

The system treats PostgreSQL as the system of record.

---

# 60. Error Contract

Phase 11 standardized the external error response shape.

Example:

```json
{
  "status": "error",
  "detail": "Human-readable explanation",
  "error": {
    "code": "ERROR_CODE",
    "message": "Safe error message",
    "request_id": "..."
  }
}
```

This provides:

- stable machine-readable code
- safe user-facing information
- request correlation

---

# 61. Safe Error Messages

External errors should not expose:

- database passwords
- provider secrets
- raw filesystem paths
- stack traces
- sensitive document contents

Operational detail belongs in protected logs, not public responses.

---

# 62. Production Documentation

Phase 11 added documentation around:

```text
Architecture
Deployment
Operations
Security
Monitoring
Backup / Restore
Shutdown
Release Readiness
```

The repository therefore became more self-describing.

Operational knowledge no longer depended entirely on developer memory.

---

# 63. Deployment Runbook

A production runbook documents activities such as:

```text
Configure Environment
      ↓
Verify Database
      ↓
Apply Migrations
      ↓
Start API
      ↓
Start Worker
      ↓
Start Proxy
      ↓
Check Health
      ↓
Verify Application
```

This reduces manual deployment ambiguity.

---

# 64. Security Documentation

Security documentation describes boundaries such as:

```text
Reviewer Identity
Trusted Proxy
Authorization
CORS
Security Headers
Rate Limiting
Secrets
Storage
```

Documenting the boundary is important because security assumptions hidden only in code are difficult to operate correctly.

---

# 65. Container Validation

The repository included static and deterministic tests covering container/deployment configuration.

The tested areas included:

```text
Dockerfile contracts
Compose configuration
Nginx configuration
Role behavior
Security expectations
Storage configuration
```

The Docker runtime itself still requires execution on a host where Docker is available.

---

# 66. Why Static Container Tests Still Matter

Even without starting a container, tests can detect mistakes such as:

- missing files
- wrong paths
- incorrect service configuration
- exposed ports
- missing volume declarations
- inconsistent environment settings
- reverse proxy configuration errors

These checks reduce the number of failures discovered only after deployment.

---

# 67. Production Authentication Boundary

VIGILOX intentionally does not pretend that local reviewer mode is enterprise authentication.

The architecture defines where an external identity system would integrate:

```text
Identity Provider
      ↓
Trusted Proxy
      ↓
Verified Reviewer Headers
      ↓
VIGILOX
```

This allows the document system to remain focused on document intelligence while delegating authentication to the appropriate infrastructure layer.

---

# 68. Security Inventory

Phase 11 verification covered security areas including:

```text
Identity spoofing
Trusted proxy configuration
Authorization
Secrets handling
Request IDs
CORS
Security headers
Rate limiting
Storage paths
TLS material
Error leakage
Logging privacy
```

The purpose was to verify that the security design existed across the application rather than as isolated middleware.

---

# 69. Test Coverage Expansion

The production-hardening phase significantly expanded the deterministic regression suite.

Tests covered areas such as:

- API error contracts
- reviewer identity
- trusted proxy behavior
- production startup validation
- storage safety
- rate limiting
- CORS
- security headers
- migrations
- container configuration
- Nginx configuration
- observability
- backup/restore tooling
- graceful shutdown
- deployment documentation

---

# 70. Repository Structure Audit

The architecture was also checked for dependency direction.

An important layering rule remained:

```text
backend/
     ↓
database/
```

but not:

```text
database/
     ↓
backend/
```

The database package remains a lower-level persistence layer.

This avoids circular architectural coupling.

---

# 71. Production Dependency Direction

The intended architecture remained:

```text
FastAPI / API
      ↓
Application Services
      ↓
Database Repositories
      ↓
Database Models
      ↓
PostgreSQL
```

Maintaining this direction makes security, testing, and deployment behavior easier to reason about.

---

# 72. Browser Acceptance Testing

Phase 11 also included real browser validation of the production-oriented frontend.

This was important because static tests cannot fully verify:

- JavaScript execution
- browser caching
- CSP behavior
- image rendering
- responsive layout
- interactive evidence overlays

---

# 73. Source Image Browser Issue

During browser validation, a source-image rendering issue was discovered.

The local frontend code had moved to direct same-origin image URLs, but the browser was still executing an older cached JavaScript file.

The stale implementation used:

```text
fetch()
   ↓
Blob
   ↓
createObjectURL()
   ↓
blob: URL
```

The active Content Security Policy did not allow that blob image source.

---

# 74. Diagnosing the Source Image Issue

The investigation compared:

```text
Local JavaScript Source
```

with:

```text
Fresh HTTP JavaScript Response
```

The files matched.

This showed that the server was serving the correct code.

The problem was the browser cache.

---

# 75. Browser Cache Resolution

After disabling cache and performing a hard reload, the browser executed the current JavaScript implementation.

The source panel correctly loaded the image directly through:

```text
/api/v1/documents/{document_id}/image
```

This validated:

- source persistence
- image endpoint
- frontend rendering
- CSP compatibility

---

# 76. Evidence Overlay Browser Validation

The browser acceptance workflow also verified evidence overlays against real rendered document dimensions.

The workspace successfully displayed OCR evidence boxes aligned with the original source.

This validated the full chain:

```text
OCR Coordinates
      ↓
Persisted Evidence
      ↓
Document API
      ↓
Frontend Scaling
      ↓
Browser Overlay
```

---

# 77. Deployment Configuration vs Actual Runtime

Phase 11 distinguishes between:

```text
Deployment Configuration Exists
```

and:

```text
Deployment Runtime Verified
```

Docker, Compose, and Nginx configuration were implemented and covered by deterministic/static tests.

The final Docker runtime build was not performed in the local development environment where Docker was unavailable.

This distinction is important for accurate release reporting.

---

# 78. Phase 11 Architecture

```text
                       Internet
                          │
                          ▼
                 ┌─────────────────┐
                 │      Nginx      │
                 │                 │
                 │ Rate Limiting   │
                 │ Header Control  │
                 │ Security Edge   │
                 └────────┬────────┘
                          │
                          ▼
                 ┌─────────────────┐
                 │     FastAPI     │
                 │                 │
                 │ API             │
                 │ Review          │
                 │ Dashboard       │
                 │ Health          │
                 │ Metrics         │
                 └────────┬────────┘
                          │
                          ▼
                ┌─────────────────────┐
                │     PostgreSQL      │
                │                     │
                │ Documents           │
                │ Jobs                │
                │ Reviews             │
                │ Audit Events        │
                │ Worker Heartbeats   │
                └──────────┬──────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │     Worker      │
                  │                 │
                  │ PaddleOCR       │
                  │ Groq            │
                  │ Validation      │
                  └─────────────────┘

        Managed Storage      Pending Storage
```

---

# 79. What Phase 11 Did Not Claim

Phase 11 deliberately did not claim:

- that local reviewer identity was production SSO
- that process-local rate limiting was globally distributed
- that API readiness proved Groq availability
- that API health proved worker health
- that Docker configuration alone proved a successful runtime deployment
- that public headers were trustworthy without a proxy boundary
- that logs should contain full OCR or document data
- that all infrastructure secrets belonged in the repository

---

# 80. Lessons Learned

## 80.1 Security Configuration Must Fail Loudly

A service that refuses unsafe startup is easier to diagnose than one that silently records bad audit identities.

---

## 80.2 Trusted Headers Require Trusted Senders

The value of an identity header is meaningless without validating who supplied it.

---

## 80.3 Reverse Proxy Security Must Be Reinforced by the Application

Infrastructure assumptions can fail.

Application-level source validation adds another security layer.

---

## 80.4 Liveness, Readiness and Worker Health Are Different Questions

A single green health endpoint cannot accurately represent every service dependency.

---

## 80.5 Production Logs Should Help Operations Without Copying Sensitive Data

Observability and privacy are not opposites.

Useful operational events can be logged without recording document contents.

---

## 80.6 Database Capacity Must Be Designed with Application Concurrency

Connection pool configuration is not an isolated database setting.

It must be considered together with API process count and thread concurrency.

---

## 80.7 Deployment Configuration Is Part of the Application

Dockerfiles, reverse proxy rules, migrations, environment templates, and runbooks are all part of production behavior.

---

## 80.8 Browser Testing Finds Problems Static Tests Cannot

Caching, CSP, image rendering, and JavaScript behavior require a real browser environment.

---

# 81. Phase 11 Deliverables

Phase 11 produced:

- production environment mode
- server-authoritative reviewer identity
- local development reviewer mode
- trusted-header reviewer mode
- trusted reverse proxy validation
- reverse proxy identity-header stripping
- backend authorization enforcement
- server-authoritative request IDs
- structured API error contracts
- security headers
- CSP
- configurable HSTS
- controlled CORS
- application-level upload rate limiting
- Nginx ingress rate limiting
- explicit request concurrency
- database pool configuration
- database connect timeout
- Alembic migration workflow
- Dockerfile
- Docker Compose configuration
- API / worker / migrate roles
- non-root container execution
- Nginx reverse proxy
- separate managed/pending storage volumes
- TLS secret exclusion
- structured operational logging
- Prometheus-compatible metrics
- API liveness
- API readiness
- worker heartbeat
- worker health state
- backup/restore procedures
- graceful shutdown behavior
- production runbook
- security documentation
- deployment documentation
- browser acceptance validation
- deployment configuration tests

---

# 82. Final Outcome

Phase 11 transformed VIGILOX from a robust application into a system with an explicit production-operating model.

Before the phase, the architecture mainly answered:

```text
How does VIGILOX process a document?
```

After Phase 11, it also answered:

```text
Who is allowed to review it?

Which requests are trusted?

How are secrets configured?

How are migrations applied?

How is the API exposed?

How is load bounded?

How is a worker monitored?

How are failures correlated?

How is state backed up?

How does the system shut down?

How is deployment verified?
```

The production-oriented system became:

```text
Secure Boundary
      +
Document Intelligence
      +
Durable Processing
      +
Human Review
      +
Observability
      +
Operational Procedures
```

This established the foundation required for the final verification and release-readiness work in Phase 12.

```text
Phase 11
Production Hardening
Security
Observability
Deployment Architecture
        ↓
Phase 12
Final Verification
Regression Gate
Release Readiness
Browser Acceptance
```

---

## Phase 11 Summary

| Area | Result |
|---|---|
| Production Environment Validation | Implemented |
| Reviewer Identity Service | Implemented |
| Trusted Header Identity | Implemented |
| Trusted Proxy Validation | Implemented |
| Backend Authorization | Implemented |
| Request IDs | Implemented |
| Structured Error Contract | Implemented |
| Security Headers | Implemented |
| Content Security Policy | Implemented |
| CORS Controls | Implemented |
| HSTS Configuration | Implemented |
| Upload Rate Limiting | Implemented |
| Request Concurrency Control | Implemented |
| Database Pool Configuration | Implemented |
| Alembic Migrations | Implemented |
| Dockerfile | Implemented |
| Docker Compose | Implemented |
| Nginx Reverse Proxy | Implemented |
| Non-Root Container Runtime | Implemented |
| Structured Logging | Implemented |
| Operational Metrics | Implemented |
| API Liveness | Implemented |
| API Readiness | Implemented |
| Worker Heartbeat | Implemented |
| Worker Health | Implemented |
| Backup / Restore Procedures | Implemented |
| Graceful Shutdown | Implemented |
| Production Runbook | Implemented |
| Security Documentation | Implemented |
| Browser Acceptance Validation | Completed |
| Docker Runtime Build | Not executed in the local development environment |

---

**Next:** `Phase 12 — Final Verification and Release Readiness`